<a href="https://colab.research.google.com/github/bhagyoday-j/ML-Assignments/blob/main/TY-open-elective/Assingments/3_Student_Placement_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment:3
**Logistic Regression** :- Student Placement Prediction

In [9]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ruchikakumbhar/placement-prediction-dataset")

print("Path to dataset files:", path)

100%|██████████| 99.7k/99.7k [00:00<00:00, 26.9MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/ruchikakumbhar/placement-prediction-dataset/versions/1


In [10]:
import os
import pandas as pd

# List files in the dataset directory
print(os.listdir(path))

['placementdata.csv']


In [11]:
csv_path = os.path.join(path, "placementdata.csv")
df = pd.read_csv(csv_path)

In [12]:
# First 5 rows
print(df.head())

# Shape of the dataset
print("\nShape of dataset : ")
print(df.shape)

# Column names and data types
print("\nInfo of dataset : ")
print(df.info())

# Check for missing values
print("\nMissing values in dataset : ")
print(df.isnull().sum())

# Summary statistics
print("\nSummary statistics of dataset : ")
print(df.describe())

   StudentID  CGPA  Internships  Projects  Workshops/Certifications  \
0          1   7.5            1         1                         1   
1          2   8.9            0         3                         2   
2          3   7.3            1         2                         2   
3          4   7.5            1         1                         2   
4          5   8.3            1         2                         2   

   AptitudeTestScore  SoftSkillsRating ExtracurricularActivities  \
0                 65               4.4                        No   
1                 90               4.0                       Yes   
2                 82               4.8                       Yes   
3                 85               4.4                       Yes   
4                 86               4.5                       Yes   

  PlacementTraining  SSC_Marks  HSC_Marks PlacementStatus  
0                No         61         79       NotPlaced  
1               Yes         78         82   

In [15]:
#Drop Unnecessary Columns
df = df.drop("StudentID", axis=1)

KeyError: "['StudentID'] not found in axis"

In [18]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["ExtracurricularActivities"] = le.fit_transform(df["ExtracurricularActivities"])
df["PlacementTraining"] = le.fit_transform(df["PlacementTraining"])
df["PlacementStatus"] = le.fit_transform(df["PlacementStatus"])

In [19]:
df.head()

,CGPA,Internships,Projects,Workshops/Certifications,AptitudeTestScore,SoftSkillsRating,ExtracurricularActivities,PlacementTraining,SSC_Marks,HSC_Marks,PlacementStatus
0,7.5,1,1,1,65,4.4,0,0,61,79,0
1,8.9,0,3,2,90,4.0,1,1,78,82,1
2,7.3,1,2,2,82,4.8,1,0,79,80,0
3,7.5,1,1,2,85,4.4,1,1,81,80,1
4,8.3,1,2,2,86,4.5,1,1,74,88,1


In [21]:
print(df["PlacementStatus"].unique())

[0 1]


In [22]:
X = df.drop("PlacementStatus", axis=1)
y = df["PlacementStatus"]

In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [24]:
#Scale Features
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [27]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()

model.fit(X_train, y_train)

LogisticRegression()

In [28]:
y_pred = model.predict(X_test)

In [29]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.8085


In [30]:
from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[966 195]
 [188 651]]
              precision    recall  f1-score   support

           0       0.84      0.83      0.83      1161
           1       0.77      0.78      0.77       839

    accuracy                           0.81      2000
   macro avg       0.80      0.80      0.80      2000
weighted avg       0.81      0.81      0.81      2000



In [31]:
from sklearn.metrics import roc_auc_score

y_prob = model.predict_proba(X_test)[:,1]

auc = roc_auc_score(y_test, y_prob)

print("ROC-AUC:", auc)

ROC-AUC: 0.8837199036217802


In [32]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
})

print(importance.sort_values("Coefficient", ascending=False))

                     Feature  Coefficient
4          AptitudeTestScore     0.610808
7          PlacementTraining     0.391990
6  ExtracurricularActivities     0.360371
8                  SSC_Marks     0.282628
5           SoftSkillsRating     0.263450
0                       CGPA     0.257470
2                   Projects     0.254356
9                  HSC_Marks     0.221181
3   Workshops/Certifications     0.095102
1                Internships    -0.010932
